In [1]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
path = kagglehub.dataset_download("pythonafroz/medquad-medical-question-answer-for-ai-research")
print("Downloaded to:", path)

100%|██████████| 4.95M/4.95M [00:00<00:00, 26.1MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/versions/1


In [3]:
import os
import pandas as pd

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".csv")]
print("CSV files found:", csv_candidates)

df = pd.read_csv(csv_candidates[0])
print()
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
print(df.head(3).to_string())

CSV files found: ['/root/.cache/kagglehub/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/versions/1/medquad.csv']

Shape: (16412, 4)
Columns: ['question', 'answer', 'source', 'focus_area']

                              question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [4]:
def find_column(df, keywords):
    matches = [c for c in df.columns if any(k.lower() in c.lower() for k in keywords)]
    if not matches:
        raise ValueError(f"No column found matching {keywords} among {df.columns.tolist()}")
    if len(matches) > 1:
        print(f"WARNING: multiple columns matched {keywords}: {matches} — using the first one, verify this is right")
    return matches[0]

question_col = find_column(df, ["question"])
answer_col = find_column(df, ["answer"])
print("Question column:", question_col)
print("Answer column:", answer_col)

print()
print("Nulls in question:", df[question_col].isna().sum())
print("Nulls in answer:", df[answer_col].isna().sum())

df = df.dropna(subset=[question_col, answer_col]).reset_index(drop=True)
print("After dropping nulls:", df.shape)

Question column: question
Answer column: answer

Nulls in question: 0
Nulls in answer: 5
After dropping nulls: (16407, 4)


In [5]:
!pip install -q nltk

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

import re
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

_stemmer = PorterStemmer()
_stop_words = set(stopwords.words("english"))

def stem_and_clean(text: str) -> str:
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text.lower())
    tokens = word_tokenize(text)
    stemmed = [_stemmer.stem(t) for t in tokens if len(t) > 1 and t not in _stop_words]
    return " ".join(stemmed)

# Sanity check the exact bug this fixes, before trusting it on the real corpus
test = stem_and_clean("How do I treat my migraines?")
print("Preprocessed test string:", repr(test))

Preprocessed test string: 'treat migrain'


In [6]:
dupe_count = df.duplicated(subset=[question_col]).sum()
print(f"Duplicate questions: {dupe_count}")
df = df.drop_duplicates(subset=[question_col]).reset_index(drop=True)
print(f"Final corpus size: {df.shape}")

questions = df[question_col].astype(str).tolist()
answers = df[answer_col].astype(str).tolist()

Duplicate questions: 1428
Final corpus size: (14979, 4)


In [7]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"  # small, fast, well-established for semantic search
model = SentenceTransformer(MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
embeddings_matrix = model.encode(questions, show_progress_bar=True, batch_size=64)
print("Embeddings shape:", embeddings_matrix.shape)

Batches:   0%|          | 0/235 [00:00<?, ?it/s]

Embeddings shape: (14979, 384)


In [9]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def test_retrieve(query, top_k=3):
    query_vec = model.encode([query])
    sims = cosine_similarity(query_vec, embeddings_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    for i in top_idx:
        print(f"  score={sims[i]:.3f}  Q: {questions[i]}")

print("Test 1: plural/verb-form robustness (v1 handled this fine already)")
test_retrieve("How do I treat my migraines?")

print()
print("Test 2: THE FAILURE CASE — v1 scored this 0.970 against 'Good syndrome'")
test_retrieve("recommend a good science fiction movie")

print()
print("Test 3: general off-topic check")
test_retrieve("what is the capital of France")

Test 1: plural/verb-form robustness (v1 handled this fine already)
  score=0.900  Q: What are the treatments for Migraine ?
  score=0.807  Q: What are the treatments for Basilar migraine ?
  score=0.764  Q: What are the treatments for sporadic hemiplegic migraine ?

Test 2: THE FAILURE CASE — v1 scored this 0.970 against 'Good syndrome'
  score=0.217  Q: What are the treatments for Schindler disease ?
  score=0.213  Q: What is the outlook for Iniencephaly ?
  score=0.211  Q: what research (or clinical trials) is being done for Iniencephaly ?

Test 3: general off-topic check
  score=0.224  Q: What is (are) Erb-Duchenne and Dejerine-Klumpke Palsies ?
  score=0.223  Q: What is the outlook for Erb-Duchenne and Dejerine-Klumpke Palsies ?
  score=0.202  Q: What are the symptoms of Dermochondrocorneal dystrophy of Franois ?


In [12]:
import json
import os

os.makedirs("chatbot_artifacts_v2", exist_ok=True)

np.save("chatbot_artifacts_v2/embeddings_matrix.npy", embeddings_matrix)

with open("chatbot_artifacts_v2/embedding_model_name.json", "w") as f:
    json.dump({"model_name": MODEL_NAME}, f)

qa_corpus = [
    {"question": questions[i], "answer": answers[i], "sources": ["MedQuAD"]}
    for i in range(len(questions))
]
with open("chatbot_artifacts_v2/qa_corpus.json", "w") as f:
    json.dump(qa_corpus, f)

print("Artifacts written:")
!ls -la chatbot_artifacts_v2

Artifacts written:
total 42952
drwxr-xr-x 2 root root     4096 Sep 27 21:10 .
drwxr-xr-x 1 root root     4096 Sep 27 21:10 ..
-rw-r--r-- 1 root root       34 Sep 27 21:10 embedding_model_name.json
-rw-r--r-- 1 root root 23007872 Sep 27 21:10 embeddings_matrix.npy
-rw-r--r-- 1 root root 20958306 Sep 27 21:10 qa_corpus.json


In [13]:
import shutil
shutil.make_archive("v2_chatbot_artifacts", "zip", "chatbot_artifacts_v2")

from google.colab import files
files.download("v2_chatbot_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>